# Classification Track — Telco Customer Churn (Part A)
**23CSE301 Machine Learning Capstone — Review 1**

**Dataset:** `data/telco_churn.csv` — customer account, service, and billing attributes for a
telecom operator.
**Target:** `Churn` (Yes/No) — binary classification problem.

This notebook covers **Part A** (Review 1 scope): data audit, EDA, cleaning, feature engineering,
and the first 5 required classification algorithms (Logistic Regression, KNN, Naive Bayes,
Decision Tree, SVM) with their evaluation. Part B (Random Forest, AdaBoost, Gradient Boosting,
Bagging, MLP) and the consolidated 10-algorithm comparison are completed for **Review 2**.


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100


## Section A — Dataset Loading & Audit

In [8]:
df = pd.read_csv(r"data\telco_churn.csv")
print('Shape:', df.shape)
df.head()


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
df.dtypes


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [10]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df.isnull().sum()
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': (missing/len(df)*100).round(2)})
missing_report[missing_report['missing_count'] > 0]


,missing_count,missing_pct
TotalCharges,11,0.16


In [11]:
print(df['Churn'].value_counts())
print((df['Churn'].value_counts(normalize=True) * 100).round(2))


Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


**Observation:** 7,043 customers, 21 columns. The target `Churn` is binary (Yes/No) and
**imbalanced** (~26.5% churn vs ~73.5% retained) — this motivates using a **stratified**
train/test split and weighted F1 / ROC-AUC rather than plain accuracy alone. `TotalCharges` had a
small number of blank entries corresponding to customers with `tenure = 0` (brand-new sign-ups who
haven't been billed yet) — a real, explainable data-quality quirk rather than random missingness.